In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "scripts" / "generate_finer_cam_panderm.py").exists():
    REPO_ROOT = REPO_ROOT.parent
import sys
sys.path.insert(0, str(REPO_ROOT))

from src.slides.slide_cam import *

HAM_ROOT  = REPO_ROOT / "data" / "HAM10000"
CKPT_DIR  = REPO_ROOT / "external" / "checkpoints5"
FIG_DIR   = REPO_ROOT / "figures" / "slides"

CKPT_HA5 = CKPT_DIR / "checkpoint-best-gap-ha5.pth"
CKPT_HA0 = CKPT_DIR / "checkpoint-best-gap-ha0.pth"

df = pd.read_csv(HAM_ROOT / "mel_nv" / "ham_mel_nv_clean.csv")
df = ensure_gt_label(df)
test = df[df["split"] == "test"].reset_index(drop=True)
# print(len(test), test["gt_label"].value_counts().to_dict())

gap5 = load_cam_model(CKPT_HA5, pooling="mean", block_index=-1, tag="HA5")

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:741: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[HA5] PanDerm Base FT pooling=mean target=blocks[11].norm1


In [2]:
# Rank test MEL cases by how similar the MEL query and NV query maps are.
cand = rank_gap_candidates(
    gap5, test[test.gt_label == "MEL"].sample(40, random_state=42),
    HAM_ROOT, n=40,
)
display(cand.head(10))

,image_id,gt_label,pred,correct,topk_iou,pearson
0,ISIC_0033644,MEL,MEL,True,0.974813,0.999189
1,ISIC_0029013,MEL,NV,False,0.969001,0.997087
2,ISIC_0026150,MEL,MEL,True,0.968615,0.997829
3,ISIC_0030552,MEL,NV,False,0.963224,0.998344
4,ISIC_0031565,MEL,MEL,True,0.959008,0.992565
5,ISIC_0029893,MEL,MEL,True,0.952909,0.985929
6,ISIC_0029698,MEL,MEL,True,0.951390,0.975220
7,ISIC_0026094,MEL,MEL,True,0.950253,0.994137
8,ISIC_0028220,MEL,MEL,True,0.938574,0.988703
9,ISIC_0030360,MEL,MEL,True,0.932602,0.975002


In [ ]:
# IMG_ID = cand.iloc[0]["image_id"]          # or hardcode e.g. "ISIC_0024308"
IMG_ID = "ISIC_0031408"
row = test[test.image_id == IMG_ID].iloc[0]

res  = compute_cams(gap5, HAM_ROOT / row["image_rel_path"])
mask = load_mask_crop(HAM_ROOT / row["mask_rel_path"])

iou = topk_iou(res["cam_gradcam"], res["cam_gradcam_B"], 10.0)
r   = pearson(res["cam_gradcam"], res["cam_gradcam_B"])

render_row(
    tiles=[
        draw_mask_contour(res["rgb"], mask),
        heat_overlay(res["cam_gradcam"],   res["rgb"], top_pct=None),
        heat_overlay(res["cam_gradcam_B"], res["rgb"], top_pct=None),
    ],
    titles=[
        "Dermoscopic image",
        '"Where is melanoma?"',
        '"Where is nevus?"',
    ],
    captions=[
        f"{IMG_ID}   ground truth {row['gt_label']}",
        "top 100 percent activation",
        f"top 100 percent activation   IoU {iou:.2f}   r {r:.2f}",
    ],
    out_path=FIG_DIR / "fig1_interpretability_gap.png",
)

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig1_interpretability_gap.png


In [4]:
gap0 = load_cam_model(CKPT_HA0, pooling="mean", block_index=-1, tag="HA0")

cand2 = rank_alignment_candidates(
    gap0, gap5, test.sample(40, random_state=7),
    HAM_ROOT, HAM_ROOT, n=40,
)
display(cand2.head(10))

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:741: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[HA0] PanDerm Base FT pooling=mean target=blocks[11].norm1


,image_id,gt_label,mask_area_frac,inside_ha0,inside_ha5,delta,pred_ha0,pred_ha5
0,ISIC_0026706,NV,0.152802,0.001590,0.867766,0.866176,MEL,MEL
1,ISIC_0027895,NV,0.295241,0.108314,0.966719,0.858405,NV,NV
2,ISIC_0030738,NV,0.144352,0.024079,0.880865,0.856785,NV,NV
3,ISIC_0032083,NV,0.138054,0.007850,0.838335,0.830485,NV,NV
4,ISIC_0028911,NV,0.115214,0.012330,0.838292,0.825962,NV,NV
5,ISIC_0028296,NV,0.224510,0.063118,0.864985,0.801867,NV,NV
6,ISIC_0029224,NV,0.209782,0.093403,0.881361,0.787958,NV,NV
7,ISIC_0025902,NV,0.199896,0.082803,0.855859,0.773057,MEL,MEL
8,ISIC_0025214,NV,0.378827,0.135501,0.890376,0.754875,MEL,MEL
9,ISIC_0032779,NV,0.121074,0.045355,0.799208,0.753853,NV,NV


In [5]:
# IMG_ID2 = cand2.iloc[0]["image_id"]
# row2 = test[test.image_id == IMG_ID2].iloc[0]
# gt   = row2["gt_label"]
# ref  = "NV" if gt == "MEL" else "MEL"

# mask = load_mask_crop(HAM_ROOT / row2["mask_rel_path"])
# r0 = compute_cams(gap0, HAM_ROOT / row2["image_rel_path"], a_class=gt, b_class=ref)
# r5 = compute_cams(gap5, HAM_ROOT / row2["image_rel_path"], a_class=gt, b_class=ref)

# in0 = cam_mass_inside(r0["cam_gradcam"], mask)
# in5 = cam_mass_inside(r5["cam_gradcam"], mask)

# render_row(
#     tiles=[
#         draw_mask_contour(r5["rgb"], mask),
#         draw_mask_contour(heat_overlay(r0["cam_gradcam"], r0["rgb"]), mask),
#         draw_mask_contour(heat_overlay(r5["cam_gradcam"], r5["rgb"]), mask),
#     ],
#     titles=[
#         "Image + lesion boundary",
#         "No alignment   lambda = 0",
#         "With alignment   lambda = 5",
#     ],
#     captions=[
#         f"{IMG_ID2}   ground truth {gt}",
#         f"{in0*100:.0f} percent of activation inside lesion",
#         f"{in5*100:.0f} percent of activation inside lesion",
#     ],
#     out_path=FIG_DIR / "fig2_alignment_effect.png",
# )

sub = stratified_subset(test, n_per_class=70, seed=0)
dist = rank_alignment_candidates(gap0, gap5, sub, HAM_ROOT, HAM_ROOT, n=len(sub))

for cls in ["NV", "MEL"]:
    d = dist[dist.gt_label == cls]
    med = d.iloc[(d.delta - d.delta.median()).abs().argsort()].iloc[0]
    row = test[test.image_id == med.image_id].iloc[0]
    gt, ref = cls, ("NV" if cls == "MEL" else "MEL")

    mask = load_mask_crop(HAM_ROOT / row["mask_rel_path"])
    r0 = compute_cams(gap0, HAM_ROOT / row["image_rel_path"], a_class=gt, b_class=ref)
    r5 = compute_cams(gap5, HAM_ROOT / row["image_rel_path"], a_class=gt, b_class=ref)

    render_row(
        tiles=[
            draw_mask_contour(r5["rgb"], mask),
            draw_mask_contour(heat_overlay(r0["cam_gradcam"], r0["rgb"]), mask),
            draw_mask_contour(heat_overlay(r5["cam_gradcam"], r5["rgb"]), mask),
        ],
        titles=["Image + lesion boundary",
                "No alignment   lambda = 0",
                "With alignment   lambda = 5"],
        captions=[
            f"{med.image_id}   ground truth {gt}   median case",
            f"{med.inside_ha0*100:.0f} percent of activation inside lesion",
            f"{med.inside_ha5*100:.0f} percent of activation inside lesion",
        ],
        out_path=FIG_DIR / f"fig2_alignment_{cls.lower()}.png",
    )

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig2_alignment_nv.png
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig2_alignment_mel.png


In [6]:
IMG_ID3 = IMG_ID
row3 = test[test.image_id == IMG_ID3].iloc[0]
res3 = compute_cams(gap5, HAM_ROOT / row3["image_rel_path"])
mask3 = load_mask_crop(HAM_ROOT / row3["mask_rel_path"])
cam3 = res3["cam_gradcam"]

render_row(
    tiles=[
        draw_mask_contour(res3["rgb"], mask3),
        heat_overlay(cam3, res3["rgb"]),
        heat_overlay(cam3, res3["rgb"], top_pct=10),
    ],
    titles=["Image + lesion boundary", "Full CAM", "Top 10 percent only"],
    captions=[
        f"{IMG_ID3}   ground truth {row3['gt_label']}",
        "covers most of the lesion",
        "the region compared against clinician annotation",
    ],
    out_path=FIG_DIR / "fig3_top10_definition.png",
)

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig3_top10_definition.png


In [7]:
row = test[test.image_id == IMG_ID].iloc[0]
gt  = row["gt_label"]

cam_panel(
    gap5,
    HAM_ROOT / row["image_rel_path"],
    HAM_ROOT / row["mask_rel_path"],
    a_class=gt, b_class="NV" if gt == "MEL" else "MEL",
    alpha=0.8, top_pct=10,
    image_id=IMG_ID, gt_label=gt,
    out_path=FIG_DIR / "fig4_cam_comparison.png",
)

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig4_cam_comparison.png


In [8]:
# A. Preprocessing patch took effect. Compare to outputs/ha_sweep/<run>/log.txt
sub = stratified_subset(test, n_per_class=70, seed=0)
print(sub.gt_label.value_counts().to_dict())
for cm in (gap0, gap5):
    print(quick_classification(cm, sub, HAM_ROOT))

# B. Full distribution, stratified. Replaces the cherry pick.
dist = rank_alignment_candidates(gap0, gap5, sub, HAM_ROOT, HAM_ROOT, n=len(sub))
for cls in ["NV", "MEL"]:
    d = dist[dist.gt_label == cls]
    med = d.iloc[(d.delta - d.delta.median()).abs().argsort()].iloc[0]
    print(cls, med.image_id, round(med.delta, 3),
          round(med.inside_ha0, 3), round(med.inside_ha5, 3))
# print(dist[["inside_ha0", "inside_ha5", "mask_area_frac", "delta"]].describe().round(3))

# C. Per class. Confirms the effect is not NV only.
print(dist.groupby("gt_label")[["inside_ha0", "inside_ha5", "delta"]].median().round(3))

# D. Below or above chance.
print("HA0 below chance:", round((dist.inside_ha0 < dist.mask_area_frac).mean(), 3))
print("HA5 above chance:", round((dist.inside_ha5 > dist.mask_area_frac).mean(), 3))

# E. Median case for the slide instead of the extreme.
med = dist.iloc[(dist.delta - dist.delta.median()).abs().argsort()].iloc[0]
print("median case:", med.image_id, med.gt_label, round(med.delta, 3))

# F. Fixed hotspot or content driven.
m0 = mean_cam_map(gap0, sub.head(60), HAM_ROOT)
m5 = mean_cam_map(gap5, sub.head(60), HAM_ROOT)
white = np.ones((224, 224, 3), np.float32)
render_row(
    tiles=[heat_overlay(m0, white), heat_overlay(m5, white)],
    titles=["Mean CAM, lambda 0", "Mean CAM, lambda 5"],
    captions=["averaged over 60 images", "averaged over 60 images"],
    out_path=FIG_DIR / "diag_mean_cam.png",
)

{'MEL': 70, 'NV': 70}
{'tag': 'HA0', 'n': 140, 'auc': 0.9777551020408163, 'bacc': 0.9071428571428571}
{'tag': 'HA5', 'n': 140, 'auc': 0.9622448979591836, 'bacc': 0.8857142857142857}
NV ISIC_0024633 0.692 0.034 0.726
MEL ISIC_0028965 0.348 0.57 0.917
          inside_ha0  inside_ha5  delta
gt_label                               
MEL            0.502       0.887  0.344
NV             0.100       0.859  0.691
HA0 below chance: 0.486
HA5 above chance: 1.0
median case: ISIC_0029430 NV 0.496
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/diag_mean_cam.png


In [9]:
nv = sub[sub.gt_label == "NV"]
bg0 = mean_background_cam(gap0, nv, HAM_ROOT, HAM_ROOT)
bg5 = mean_background_cam(gap5, nv, HAM_ROOT, HAM_ROOT)
white = np.ones((224, 224, 3), np.float32)
render_row(
    tiles=[heat_overlay(bg0, white), heat_overlay(bg5, white)],
    titles=["NV background heat, lambda 0", "NV background heat, lambda 5"],
    captions=[f"lesion masked out, n={len(nv)}", f"lesion masked out, n={len(nv)}"],
    out_path=FIG_DIR / "diag_background_cam.png",
)

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/diag_background_cam.png


In [10]:
sub = stratified_subset(test, n_per_class=70, seed=0)   # all 70 MEL, 70 of 951 NV

res = pd.concat([
    sweep_blocks(CKPT_DIR / "checkpoint-best-gap-ha5.pth", "mean",
                 sub, HAM_ROOT, HAM_ROOT, tag="GAP_HA5"),
    sweep_blocks(CKPT_DIR / "checkpoint-best-cls-ha5.pth", "cls",
                 sub, HAM_ROOT, HAM_ROOT, tag="CLS_HA5"),
])
# res.to_csv(OUT_ROOT / "block_sweep.csv", index=False)

g = res[res.cam_method == "gradcam_a"]
by_class = (g.groupby(["model", "gt_label", "block"])
              [["top10_inside", "pointing_game", "top10_minus_random"]]
              .mean().round(3))
print(by_class)

# Balanced across classes, this is the selection table
bal = by_class.groupby(["model", "block"]).mean().round(3)
print(bal.sort_values(["model", "top10_inside"], ascending=[True, False]))

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:741: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[GAP_HA5] PanDerm Base FT pooling=mean target=blocks[11].norm1
[GAP_HA5] block -1 -> blocks[11].norm1
[GAP_HA5] block -2 -> blocks[10].norm1
[GAP_HA5] block -4 -> blocks[8].norm1
[GAP_HA5] block -6 -> blocks[6].norm1
[GAP_HA5] block -8 -> blocks[4].norm1
[GAP_HA5] block -10 -> blocks[2].norm1
[GAP_HA5] block -12 -> blocks[0].norm1


/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:741: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[CLS_HA5] PanDerm Base FT pooling=cls target=blocks[11].norm1
[CLS_HA5] block -1 -> blocks[11].norm1
[CLS_HA5] block -2 -> blocks[10].norm1
[CLS_HA5] block -4 -> blocks[8].norm1
[CLS_HA5] block -6 -> blocks[6].norm1
[CLS_HA5] block -8 -> blocks[4].norm1
[CLS_HA5] block -10 -> blocks[2].norm1
[CLS_HA5] block -12 -> blocks[0].norm1
                        top10_inside  pointing_game  top10_minus_random
model   gt_label block                                                 
CLS_HA5 MEL      -12           0.129          0.143              -0.245
                 -10           0.249          0.243              -0.125
                 -8            0.281          0.286              -0.093
                 -6            0.749          0.686               0.375
                 -4            0.355          0.371              -0.018
                 -2            0.149          0.114              -0.224
                 -1            0.974          1.000               0.600
        NV       -12

In [11]:
res0 = sweep_blocks(CKPT_DIR / "checkpoint-best-gap-ha0.pth", "mean",
                    sub, HAM_ROOT, HAM_ROOT, tag="GAP_HA0")

g0 = res0[res0.cam_method == "gradcam_a"]
print(g0.groupby(["gt_label", "block"])[["top10_inside", "pointing_game"]].mean().round(3))

/storage/homefs/cn21m021/projects/master-thesis/scripts/generate_finer_cam_panderm.py:741: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, m

[GAP_HA0] PanDerm Base FT pooling=mean target=blocks[11].norm1
[GAP_HA0] block -1 -> blocks[11].norm1
[GAP_HA0] block -2 -> blocks[10].norm1
[GAP_HA0] block -4 -> blocks[8].norm1
[GAP_HA0] block -6 -> blocks[6].norm1
[GAP_HA0] block -8 -> blocks[4].norm1
[GAP_HA0] block -10 -> blocks[2].norm1
[GAP_HA0] block -12 -> blocks[0].norm1
                top10_inside  pointing_game
gt_label block                             
MEL      -12           0.147          0.157
         -10           0.244          0.314
         -8            0.216          0.214
         -6            0.668          0.700
         -4            0.649          0.714
         -2            0.428          0.514
         -1            0.681          0.757
NV       -12           0.473          0.571
         -10           0.688          0.757
         -8            0.422          0.486
         -6            0.115          0.100
         -4            0.109          0.100
         -2            0.542          0.657
       

In [12]:
res_all = pd.concat([res, res0])

ORDER  = ["GAP_HA0", "GAP_HA5", "CLS_HA5"]
LABELS = {"GAP_HA0": "No alignment (GAP)",
          "GAP_HA5": "With alignment (GAP)",
          "CLS_HA5": "With alignment (CLS)"}

plot_block_sensitivity(res_all, order=ORDER, titles=LABELS,
                       out_path=FIG_DIR / "fig5_block_sensitivity.png")

plot_worst_class(res_all, order=ORDER, labels=LABELS,
                 out_path=FIG_DIR / "fig5b_worst_class.png")

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig5_block_sensitivity.png
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig5b_worst_class.png


In [13]:
pred = predict_frame(gap5, test, HAM_ROOT)
m = performance_panel(pred, "Melanoma vs nevus, HAM10000 test set",
                      out_path=FIG_DIR / "fig6_performance.png")
print(m)

# The dx_type slide
nv = pred[pred.gt_label == "NV"]
fa = (nv.groupby("dx_type")
        .agg(n=("pred", "size"),
             false_alarms=("pred", lambda s: (s == "MEL").sum()))
        .assign(rate=lambda d: (d.false_alarms / d.n).round(3)))
print(fa)

mel = pred[pred.gt_label == "MEL"]
print(mel.dx_type.value_counts())

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig6_performance.png
{'sens': np.float64(0.9428571428571428), 'spec': np.float64(0.8464773922187171), 'ppv': np.float64(0.3113207547169811), 'auc': 0.9623103500075109, 'bacc': np.float64(0.8946672675379299), 'tp': np.int64(66), 'fn': np.int64(4), 'fp': np.int64(146), 'tn': np.int64(805)}
             n  false_alarms   rate
dx_type                            
consensus   63            32  0.508
follow_up  740            12  0.016
histo      148           102  0.689
dx_type
histo    70
Name: count, dtype: int64


In [14]:
def subset_metrics(pred: pd.DataFrame, name: str = "") -> dict:
    y = (pred.gt_label == "MEL").astype(int).values
    yh = (pred.pred == "MEL").astype(int).values
    tp = int(((y == 1) & (yh == 1)).sum())
    fn = int(((y == 1) & (yh == 0)).sum())
    fp = int(((y == 0) & (yh == 1)).sum())
    tn = int(((y == 0) & (yh == 0)).sum())
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    return {"subset": name, "n": len(pred), "n_mel": tp + fn, "n_nv": tn + fp,
            "sens": round(sens, 3), "spec": round(spec, 3),
            "bacc": round(0.5 * (sens + spec), 3), "fn": fn, "fp": fp}

print(pd.DataFrame([
    subset_metrics(pred, "all test"),
    subset_metrics(pred[pred.dx_type == "histo"], "biopsied only"),
    subset_metrics(pred[pred.dx_type != "follow_up"], "excl. follow_up"),
]))

            subset     n  n_mel  n_nv   sens   spec   bacc  fn   fp
0         all test  1021     70   951  0.943  0.846  0.895   4  146
1    biopsied only   218     70   148  0.943  0.311  0.627   4  102
2  excl. follow_up   281     70   211  0.943  0.365  0.654   4  134


In [15]:
sets = {s: set(df[df.split == s].lesion_id) for s in df.split.unique()}
print("train/test lesion overlap:", len(sets["train"] & sets["test"]))
print("val/test lesion overlap:  ", len(sets["val"] & sets["test"]))

train/test lesion overlap: 0
val/test lesion overlap:   0


In [16]:
print(pd.crosstab(test.dx_type, test.dataset, margins=True))

m = pred.merge(test[["image_id", "dataset", "lesion_id", "localization", "age"]],
               on="image_id")
print(m[m.gt_label == "NV"]
        .groupby(["dataset", "dx_type"])["correct"]
        .agg(n="size", err=lambda s: (~s).sum())
        .assign(rate=lambda d: (d.err / d.n).round(3)))

dataset    rosendahl  vidir_modern  vidir_molemax  vienna_dias   All
dx_type                                                             
consensus          0            63              0            0    63
follow_up          0             0            740            0   740
histo             98            69              9           42   218
All               98           132            749           42  1021
                           n  err   rate
dataset       dx_type                   
rosendahl     histo       67   52  0.776
vidir_modern  consensus   63   32  0.508
              histo       42   22  0.524
vidir_molemax follow_up  740   12  0.016
              histo        5    1  0.200
vienna_dias   histo       34   27  0.794


In [17]:
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    p = pred.assign(pred=np.where(pred.p_mel >= t, "MEL", "NV"))
    p["correct"] = p.pred == p.gt_label
    r = subset_metrics(p, f"t={t}")
    r["nv_histo_err"] = round((~p[(p.gt_label == "NV") &
                                  (p.dx_type == "histo")].correct).mean(), 3)
    rows.append(r)
print(pd.DataFrame(rows))

   subset     n  n_mel  n_nv   sens   spec   bacc  fn   fp  nv_histo_err
0   t=0.3  1021     70   951  0.986  0.739  0.862   1  248         0.899
1   t=0.4  1021     70   951  0.986  0.810  0.898   1  181         0.770
2   t=0.5  1021     70   951  0.943  0.846  0.895   4  146         0.689
3   t=0.6  1021     70   951  0.929  0.889  0.909   5  106         0.554
4   t=0.7  1021     70   951  0.857  0.920  0.889  10   76         0.426
5   t=0.8  1021     70   951  0.714  0.953  0.833  20   45         0.270
6   t=0.9  1021     70   951  0.314  0.995  0.655  48    5         0.034
7  t=0.95  1021     70   951  0.000  1.000  0.500  70    0         0.000


In [18]:
m = pred.merge(test[["image_id", "dataset", "lesion_id"]], on="image_id")
print(pd.crosstab(m.gt_label, m.dataset, margins=True))
print(pd.crosstab(m.dataset, m.pred, normalize="index").round(3))

dataset   rosendahl  vidir_modern  vidir_molemax  vienna_dias   All
gt_label                                                           
MEL              31            27              4            8    70
NV               67           105            745           34   951
All              98           132            749           42  1021
pred             MEL     NV
dataset                    
rosendahl      0.847  0.153
vidir_modern   0.598  0.402
vidir_molemax  0.020  0.980
vienna_dias    0.833  0.167


In [19]:
from sklearn.metrics import roc_auc_score

def auc_of(d):
    y = (d.gt_label == "MEL").astype(int)
    return round(roc_auc_score(y, d.p_mel), 3) if y.nunique() == 2 else np.nan

print("all test      ", auc_of(pred), len(pred))
print("biopsied only ", auc_of(pred[pred.dx_type == "histo"]),
      len(pred[pred.dx_type == "histo"]))

for src, d in m.groupby("dataset"):
    print(f"{src:16s} n={len(d):4d} mel={int((d.gt_label=='MEL').sum()):3d} auc={auc_of(d)}")

all test       0.962 1021
biopsied only  0.817 218
rosendahl        n=  98 mel= 31 auc=0.793
vidir_modern     n= 132 mel= 27 auc=0.928
vidir_molemax    n= 749 mel=  4 auc=0.936
vienna_dias      n=  42 mel=  8 auc=0.849


In [20]:
rows = []
for src, d in m.groupby("dataset"):
    r = subset_metrics(d, src)
    r["auc"] = auc_of(d)
    rows.append(r)
print(pd.DataFrame(rows))

          subset    n  n_mel  n_nv   sens   spec   bacc  fn  fp    auc
0      rosendahl   98     31    67  1.000  0.224  0.612   0  52  0.793
1   vidir_modern  132     27   105  0.926  0.486  0.706   2  54  0.928
2  vidir_molemax  749      4   745  0.500  0.983  0.741   2  13  0.936
3    vienna_dias   42      8    34  1.000  0.206  0.603   0  27  0.849


In [21]:
pred0 = predict_frame(gap0, test, HAM_ROOT)
m0 = pred0.merge(test[["image_id", "dataset"]], on="image_id")
m5 = pred.merge(test[["image_id", "dataset"]], on="image_id")

rows = []
for tag, d in [("HA0", m0), ("HA5", m5)]:
    for src, s in d.groupby("dataset"):
        r = subset_metrics(s, src)
        r["model"] = tag
        r["auc"] = auc_of(s)
        r["pred_mel_rate"] = round((s.pred == "MEL").mean(), 3)
        rows.append(r)
cmp = pd.DataFrame(rows).pivot(index="subset", columns="model")
print(cmp[["auc", "spec", "pred_mel_rate"]].round(3))

print("\nbiopsied only")
print("HA0", auc_of(m0[m0.image_id.isin(pred[pred.dx_type == 'histo'].image_id)]))
print("HA5", auc_of(pred[pred.dx_type == "histo"]))

                 auc          spec        pred_mel_rate       
model            HA0    HA5    HA0    HA5           HA0    HA5
subset                                                        
rosendahl      0.872  0.793  0.269  0.224         0.816  0.847
vidir_modern   0.951  0.928  0.505  0.486         0.591  0.598
vidir_molemax  0.981  0.936  0.983  0.983         0.021  0.020
vienna_dias    0.846  0.849  0.176  0.206         0.857  0.833

biopsied only
HA0 0.871
HA5 0.817


In [22]:
nv = m5[m5.gt_label == "NV"]
print("AUC predicting molemax vs other, from p_mel alone, NV only:",
      round(roc_auc_score((nv.dataset == "vidir_molemax").astype(int), 1 - nv.p_mel), 3))
print(nv.groupby("dataset")["p_mel"].describe()[["count", "mean", "50%"]].round(3))

AUC predicting molemax vs other, from p_mel alone, NV only: 0.955
               count   mean    50%
dataset                           
rosendahl       67.0  0.643  0.653
vidir_modern   105.0  0.508  0.505
vidir_molemax  745.0  0.140  0.106
vienna_dias     34.0  0.691  0.772


In [23]:
m5 = pred.merge(test[["image_id", "dataset"]], on="image_id")
srcs = source_panel(m5, out_path=FIG_DIR / "fig7_source_bias.png")
print(srcs.round(3))

plot_auc_paradox(auc_of(m5), srcs.auc, out_path=FIG_DIR / "fig7b_auc_paradox.png")

saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig7_source_bias.png
                   n  n_mel   prev   rate    auc
dataset                                         
vidir_molemax  749.0    4.0  0.005  0.020  0.936
vienna_dias     42.0    8.0  0.190  0.833  0.849
vidir_modern   132.0   27.0  0.205  0.598  0.928
rosendahl       98.0   31.0  0.316  0.847  0.793
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig7b_auc_paradox.png


In [24]:
# test_all = ensure_gt_label(test)
# print(test_all.gt_label.value_counts().to_dict())

# runs = [
#     ("GAP_HA0", "checkpoint-best-gap-ha0.pth", "mean"),
#     ("GAP_HA5", "checkpoint-best-gap-ha5.pth", "mean"),
#     ("CLS_HA5", "checkpoint-best-cls-ha5.pth", "cls"),
# ]

# out_csv = FIG_DIR.parent / "block_sweep_all_test.csv"
# parts = []
# for tag, ckpt, pooling in runs:
#     r = sweep_blocks(CKPT_DIR / ckpt, pooling, test_all,
#                      HAM_ROOT, HAM_ROOT, tag=tag)
#     parts.append(r)
#     pd.concat(parts).to_csv(out_csv, index=False)   # write after each model
#     print(f"{tag} done, {len(r)} rows")

# res = pd.concat(parts)


# g = res[res.cam_method == "gradcam_a"]
# print(g.groupby(["model", "gt_label", "block"])["top10_inside"].mean().round(3))

# # worst class per block, the number that matters
# wc = (g.groupby(["model", "gt_label", "block"])["top10_inside"].mean()
#         .groupby(level=["model", "block"]).min().round(3))
# print(wc.sort_values(ascending=False))


# import time
# t = time.time()
# _ = sweep_blocks(CKPT_DIR / "checkpoint-best-gap-ha0.pth", "mean",
#                  test_all.head(20), HAM_ROOT, HAM_ROOT,
#                  blocks=(-1,), tag="timing")
# per = (time.time() - t) / 20
# print(f"{per:.2f} s per image-block")
# print(f"full run estimate: {per * 1021 * 7 * 3 / 3600:.1f} h")

In [25]:
p = pred.merge(test[["image_id", "image_rel_path", "mask_rel_path"]],
               on="image_id")

p["cat"] = np.select(
    [(p.gt_label == "MEL") & (p.pred == "MEL"),
     (p.gt_label == "MEL") & (p.pred == "NV"),
     (p.gt_label == "NV")  & (p.pred == "NV"),
     (p.gt_label == "NV")  & (p.pred == "MEL")],
    ["TP MEL", "FN MEL", "TN NV", "FP MEL"],
    default="unknown")

print(p.cat.value_counts())

# Confidence used for picking a representative case per category.
p["conf"] = np.where(p.pred == "MEL", p.p_mel, 1 - p.p_mel)

for cat in ["TP MEL", "FN MEL", "TN NV", "FP MEL"]:
    s = p[p.cat == cat]
    if len(s) == 0:
        print(f"skip {cat}, none found")
        continue
    # Most confident case, so the panel shows a clear example not a borderline one.
    r = s.sort_values("conf", ascending=False).iloc[0]
    gt = r.gt_label
    print(f"{cat}: {r.image_id}  p_mel={r.p_mel:.3f}")
    cam_panel(
        gap5,
        HAM_ROOT / r.image_rel_path,
        HAM_ROOT / r.mask_rel_path,
        a_class=gt,
        b_class="NV" if gt == "MEL" else "MEL",
        top_pct=10.0,
        image_id=r.image_id,
        gt_label=gt,
        out_path=FIG_DIR / f"fig8_{cat.replace(' ', '_').lower()}.png",
    )

cat
TN NV     805
FP MEL    146
TP MEL     66
FN MEL      4
Name: count, dtype: int64
TP MEL: ISIC_0025234  p_mel=0.946
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig8_tp_mel.png
FN MEL: ISIC_0029013  p_mel=0.188
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig8_fn_mel.png
TN NV: ISIC_0031139  p_mel=0.012
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig8_tn_nv.png
FP MEL: ISIC_0026491  p_mel=0.922
saved /storage/homefs/cn21m021/projects/master-thesis/figures/slides/fig8_fp_mel.png


In [26]:
fn = p[p.cat == "FN MEL"]
print(fn[["image_id", "p_mel", "dataset"]] if "dataset" in fn else fn[["image_id", "p_mel"]])
print("masks exist:",
      [(HAM_ROOT / m).exists() for m in fn.mask_rel_path])

        image_id     p_mel
5   ISIC_0024886  0.482836
35  ISIC_0029013  0.188248
46  ISIC_0030366  0.478623
47  ISIC_0030552  0.454297
masks exist: [True, True, True, True]


In [27]:
p2 = p.merge(test[["image_id", "dataset"]], on="image_id")
print(p2[p2.cat == "FP MEL"].dataset.value_counts())
print(p2[p2.cat == "FN MEL"][["image_id", "p_mel", "dataset"]])

dataset
vidir_modern     54
rosendahl        52
vienna_dias      27
vidir_molemax    13
Name: count, dtype: int64
        image_id     p_mel        dataset
5   ISIC_0024886  0.482836   vidir_modern
35  ISIC_0029013  0.188248  vidir_molemax
46  ISIC_0030366  0.478623  vidir_molemax
47  ISIC_0030552  0.454297   vidir_modern


In [ ]:
from scipy.ndimage import distance_transform_edt

nv = sub[sub.gt_label == "NV"]
mel = sub[sub.gt_label == "MEL"]

for tag, d in [("NV", nv), ("MEL", mel)]:
    prof, k = np.zeros(60), 0
    for _, r in d.iterrows():
        p = HAM_ROOT / r.image_rel_path
        mp = HAM_ROOT / r.mask_rel_path
        if not (p.exists() and mp.exists()):
            continue
        gt = str(r.gt_label)
        res = compute_cams(gap5, p, a_class=gt, b_class="NV" if gt == "MEL" else "MEL")
        mask = load_mask_crop(mp)
        cam = np.asarray(res["cam_gradcam"], np.float32)
        cam = cam / (cam.max() + 1e-8)
        dist_out = distance_transform_edt(1 - mask)
        for i in range(60):
            band = (dist_out > i) & (dist_out <= i + 1)
            if band.sum():
                prof[i] += cam[band].mean()
        k += 1
    prof /= max(k, 1)
    print(f"{tag}  0-16px {prof[:16].mean():.3f}   16-40px {prof[16:40].mean():.3f}   40+ {prof[40:].mean():.3f}")

In [ ]:
CKPT_HA3 = CKPT_DIR / "checkpoint-best-gap-ha3.pth"
print(CKPT_HA3.exists())

gap3 = load_cam_model(CKPT_HA3, pooling="mean", block_index=-1, tag="HA3")

# NOTE: rank_alignment_candidates names its second-model column "inside_ha5"
# regardless of which model is passed. For d3 that column holds lambda 3 values.
d3 = rank_alignment_candidates(gap0, gap3, sub, HAM_ROOT, HAM_ROOT, n=len(sub))

print("localisation, median inside-lesion CAM mass")
print("  lambda 3:", d3.groupby("gt_label")["inside_ha5"].median().round(3).to_dict())
print("  lambda 5:", dist.groupby("gt_label")["inside_ha5"].median().round(3).to_dict())

print("\nclassification")
for cm in (gap0, gap3, gap5):
    print(" ", quick_classification(cm, sub, HAM_ROOT))

False


FileNotFoundError: Checkpoint not found: /storage/homefs/cn21m021/projects/master-thesis/outputs/ha_sweep/sweep_ha__mean__l3p0__ep10__lr1e5/checkpoint-best.pth